# Informe Técnico: Análisis Estadístico y Espectral de Señales EEG
**Práctica 2 - Algorítmica y Lógica Computacional**

## 1. Caracterización de la Crisis mediante Descriptores Estadísticos
De acuerdo con la teoría neurofisiológica, una crisis epiléptica se define formalmente como una actividad neuronal anormal excesiva en el cerebro. Esta actividad se caracteriza por su modo de inicio, su finalización y, fundamentalmente, por una sincronía aumentada. Para caracterizar estos estados computacionalmente, se procesó la base de datos dividiendo la señal en segmentos y centrando los datos (restando la media). Se aplicaron los siguientes descriptores:

*   **Descriptores de Variabilidad (Actividad Excesiva):** Se utilizaron la varianza (σ²) y la desviación estándar (σ) para medir la variabilidad de la señal. Los resultados demostraron un aumento drástico en la magnitud de σ durante el segmento de la crisis en comparación con el reposo, reflejando matemáticamente la "actividad neuronal excesiva".
*   **Descriptores de Relación Cruzada (Sincronía Aumentada):** Se calculó la matriz de covarianza y el coeficiente de correlación de Pearson (rango de -1 a 1) para describir la similitud espacial entre los 28 canales. Se observó un salto abrupto en la correlación al inicio del evento, evidenciando el fenómeno de "sincronía aumentada" donde distintas áreas cerebrales se acoplan en simultáneo.
*   **Autocorrelación (Patrones Rítmicos):** Mientras que en una señal aleatoria (reposo o ruido Gaussiano) la autocorrelación cae a cero instantáneamente, durante la crisis revela oscilaciones rítmicas persistentes, confirmando la presencia de patrones repetitivos intrínsecos al episodio.

---

## 2. Determinación de Umbral Dinámico y Tiempo de Retardo
Para la detección automatizada, se implementó un **umbral dinámico** delimitado por el rango matemático [min, max] del descriptor en estado de normalidad.

*   **Justificación del Rango [min, max]:** El algoritmo extrajo la desviación estándar (σ) de los 28 canales durante el bloque previo a la crisis ("Before"). Al calcular el mínimo y máximo absoluto de este bloque, se construyó una cota de normalidad estricta. Cualquier canal que rompa el límite superior (umbral_max) durante el análisis continuo denota una dispersión anómala asimilable a una crisis.
*   **Tiempo de Retardo (Escenario 2):** Aplicando este umbral sobre el bloque total mediante una ventana temporal móvil, se calculó el tiempo de retardo identificando el segundo exacto en el que la dispersión superó el umbral, brindando una métrica objetiva de la latencia del algoritmo.

---

## 3. Modelamiento Estadístico y Distribuciones (PDF)
El modelamiento permite construir una representación de los datos para validar hipótesis de clasificación.

*   **Diagrama de Cajas (Boxplot):** Las gráficas arrojaron que la "caja" (Rango Intercuartil - IQR) durante la crisis posee una dispersión significativamente mayor y una aparición masiva de outliers (valores atípicos), corroborando la inestabilidad de las amplitudes.
*   **Histograma y Ajuste de Densidad de Probabilidad (PDF):** Se comprobó que la función Normal (Gaussiana) es ineficiente para estas señales. En su lugar, la distribución **t-location-scale** se ajustó de forma óptima. Esta incorpora parámetros de posición (μ), escala (σ) y forma (ν), siendo este último indispensable para modelar las "colas pesadas" típicas de los eventos rítmicos anormales (spike-and-wave).
*   **Diagrama de Dispersión (Scatter Plot):** Al graficar los vectores de parámetros θ = (μ, σ), se observó una clara separación de clústeres: las señales normales se agruparon en la zona de baja dispersión, mientras que los valores de crisis se dispararon en el eje Y, demostrando que la desviación estándar (σ) es el parámetro discriminante primario.

---

## 4. Complejidad Algorítmica y Muestreo Aleatorio
**Análisis de Complejidad Computacional (Notación Big-O):**

1.  **Estadísticos univariados:** El cálculo de media y desviación estándar presenta una complejidad temporal de **O(N)**, ya que requiere recorrer las N muestras de forma lineal.
2.  **Relación cruzada:** La covarianza y correlación entre M canales y N muestras eleva el costo a **O(M² · N)**.
3.  **Autocorrelación:** Un enfoque tradicional en el dominio del tiempo requiere una complejidad cuadrática **θ(N²)**. Para optimizarlo, se utilizaron funciones basadas en la **Transformada Rápida de Fourier (FFT)**, lo que reduce el costo drásticamente a **θ(N log₂ N)**, permitiendo el procesamiento fluido de grandes volúmenes de datos.

**Optimización mediante Muestreo Aleatorio:** 
El uso de funciones aleatorias optimiza la carga computacional. En lugar de procesar la matriz completa de longitud N, se toma una Muestra Aleatoria Simple (SRS) de tamaño n. Dado que n << N, el costo temporal se reduce a magnitudes cercanas a **O(1)** constante. Al realizar la selección al azar, se evitan sesgos y se obtienen estimaciones fiables del umbral dinámico demandando solo una fracción de los recursos de la máquina.


In [ ]:
import numpy as np
from scipy import signal
from pyedflib import highlevel
import matplotlib.pyplot as plt
from scipy.stats import norm, t

# --- Parámetros ---
# Ruta del archivo de electroencefalograma (EEG) en formato EDF.
EDF_FILE = './archivos/chb20_12.edf' 
# Tiempos marcados en segundos donde inicia y termina la crisis epiléptica (seizure).
START_SEC = 94
END_SEC = 123
# Tamaño de la ventana de contexto (en segundos) que queremos observar antes y después de la crisis.
WINDOW_SEC = 120

# --- Carga ---
# Lee el archivo EDF. Extrae las señales (datos crudos), la información de cada canal y el encabezado general.
signals, signal_headers, header = highlevel.read_edf(EDF_FILE)
# Frecuencia de muestreo (muestras por segundo). Aquí está hardcodeada a 256 Hz.
fs = 256 

# --- Muestras ---
# Convierte los tiempos (segundos) a índices de array (muestras) multiplicando por la frecuencia de muestreo.
start_sample = START_SEC * fs
end_sample = END_SEC * fs
window_samples = WINDOW_SEC * fs
total_samples = signals.shape[1] # Número total de muestras en la señal

# Calcula dónde empiezan y terminan las ventanas de contexto.
# Usa max(0, ...) para evitar índices negativos si la crisis ocurre muy cerca del inicio.
inicio_before = max(0, start_sample - window_samples)
# Usa min(total, ...) para evitar salirnos del límite del archivo si la crisis ocurre cerca del final.
fin_after = min(total_samples, end_sample + window_samples)

# --- Segmentación y centrado ---
def extract_and_center(sig, s, e):
    """
    Extrae un fragmento de la señal y la "centra" restándole la media a cada canal.
    Esto elimina el componente de corriente continua (DC offset) o línea base.
    """
    segment = sig[:, s:e] # Extrae todas las filas (canales) y las columnas desde 's' hasta 'e'
    # Resta la media de cada canal (axis=1). keepdims=True mantiene la forma para poder restar correctamente.
    return segment - segment.mean(axis=1, keepdims=True)

# Extraemos los tres bloques de interés ya centrados en cero:
before_centered = extract_and_center(signals, inicio_before, start_sample) # Antes de la crisis
seizure_centered = extract_and_center(signals, start_sample, end_sample)   # Durante la crisis
after_centered   = extract_and_center(signals, end_sample, fin_after)      # Después de la crisis

# Une los tres segmentos consecutivamente en un solo bloque grande para análisis global.
total_block = np.concatenate(
    (before_centered, seizure_centered, after_centered), axis=1
)

# imprimo resultados iniciales
print(f"Frecuencia de muestreo: {fs} Hz")
print(f"Dimensiones del bloque 'Before': {before_centered.shape}")
print(f"Dimensiones del bloque 'Crisis': {seizure_centered.shape}")
print(f"Dimensiones del bloque 'After': {after_centered.shape}")
print(f"Dimensiones del 'Bloque Total': {total_block.shape}")


# --- Descriptores ---
def compute_stats(seg):
    """
    Calcula diversas métricas estadísticas para un segmento de señal dado.
    """
    var = np.var(seg, axis=1) # Varianza: qué tan dispersos están los datos
    return {
        'var':      var,
        'mean':     np.mean(seg,axis=1),
        'std':      np.sqrt(var), # Desviación estándar: raíz de la varianza
        'abs_mean': np.mean(np.abs(seg), axis=1), # Media de los valores absolutos (amplitud promedio)
        'cov':      np.cov(seg), # Matriz de covarianza: cómo varían los canales juntos
        'pearson':  np.corrcoef(seg), # Matriz de correlación de Pearson: relación lineal entre canales (-1 a 1)
    }

# Agrupamos los segmentos en un diccionario para procesarlos fácilmente con un bucle.
segments = {
    'before':  before_centered,
    'seizure': seizure_centered,
    'after':   after_centered,
}

# Creamos un diccionario 'stats' que calcula y guarda las métricas para cada uno de los 3 segmentos.
stats = {name: compute_stats(seg) for name, seg in segments.items()}

# --- Autocorrelación ---
C0 = 0 # Índice del canal a analizar (Canal 0)
# Calcula la autocorrelación para el canal 0 en los tres segmentos.
# Esto mide qué tan similar es la señal consigo misma al desplazarla en el tiempo.
autocorr = {
    name: signal.correlate(seg[C0], seg[C0], mode='full')
    for name, seg in segments.items()
}

# --- Reporte ---
C0, C1 = 0, 1 # Seleccionamos los canales 0 y 1 para mostrar el reporte en pantalla

print(f"fs: {fs} Hz | Bloque total: {total_block.shape}")

# Imprime la varianza, desviación estándar y media absoluta SOLO para el canal 0 (C0) en los 3 periodos.
for metric in ('var', 'std', 'abs_mean'):
    vals = " | ".join(
        f"{name}: {stats[name][metric][C0]:.2f}"
        for name in ('before', 'seizure', 'after')
    )
    print(f"{metric.upper():12} -> {vals}")

# Imprime la covarianza y la correlación entre el canal 0 (C0) y el canal 1 (C1)
# Compara el estado "antes" (before) con el estado "durante" (seizure).
print(f"\nCovarianza C{C0}-C{C1}  -> "
      f"Antes: {stats['before']['cov'][C0, C1]:.2f} | "
      f"Crisis: {stats['seizure']['cov'][C0, C1]:.2f}")
print(f"Pearson C{C0}-C{C1}     -> "
      f"Antes: {stats['before']['pearson'][C0, C1]:.2f} | "
      f"Crisis: {stats['seizure']['pearson'][C0, C1]:.2f}")


# --- ESCENARIO 1: Umbral Dinámico [min, max] de un Descriptor Estadístico ---
def escenario1paso2():
    descriptor = 'std'  # Podés cambiar a 'var','std', 'abs_mean', etc.

    # Extraemos el descriptor para todos los canales en cada bloque
    vals_before  = stats['before'][descriptor]   # shape: (n_canales,)
    vals_seizure = stats['seizure'][descriptor]
    vals_after   = stats['after'][descriptor]

    # Umbral dinámico: [min, max] del descriptor en el bloque 'before'
    umbral_min = vals_before.min()
    umbral_max = vals_before.max()

    n_canales = len(vals_before)
    x_pos = np.arange(n_canales)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True, sharey=True)
    bloques = [
        ('Antes (Reposo)',        vals_before,  'steelblue'),
        ('Crisis (Epilepsia)',    vals_seizure, 'tomato'),
        ('Después (Recuperación)',vals_after,   'seagreen'),
    ]

    for ax, (titulo, vals, color) in zip(axes, bloques):
        # Zona umbral (rango normal definido por 'before')
        ax.axhspan(umbral_min, umbral_max, color='gold', alpha=0.25, zorder=0,
                label=f'Rango Normal [{umbral_min:.2f}, {umbral_max:.2f}]')
        ax.axhline(umbral_max, color='orange', linestyle='--', linewidth=1.2)
        ax.axhline(umbral_min, color='orange', linestyle='--', linewidth=1.2)

        # Coloreamos en rojo los canales que superan el umbral
        for i, v in enumerate(vals):
            fuera = v > umbral_max or v < umbral_min
            ax.bar(i, v, color='red' if fuera else color,
                alpha=0.8, width=0.7, zorder=2)

        ax.set_title(titulo, fontsize=11, fontweight='bold')
        ax.set_ylabel(descriptor.upper(), fontsize=10)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(axis='y', linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Canal', fontsize=11)
    axes[-1].set_xticks(x_pos)
    axes[-1].set_xticklabels(x_pos, fontsize=7)

    fig.suptitle(f'Umbral Dinámico [min, max] del descriptor "{descriptor.upper()}"\n'
                f'por canal — Umbral definido por bloque "Antes"',
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


def escenario1paso4():
    canal = 0

    # 1. Extraemos las señales del Canal 0 para los 3 bloques
    data_before = before_centered[canal, :]
    data_seizure = seizure_centered[canal, :]
    data_after = after_centered[canal, :]

    # Creamos una figura con 3 subgráficos horizontales. 'axes' es un arreglo de 3 posiciones
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # --- A. DIAGRAMA DE CAJAS ---
    n_min = min(len(data_before), len(data_seizure), len(data_after)) 
                # Lo usamos ya que la crisis es mucho mas corta que el resto de bloques, haciendo que el grafico no sea tan representativo, ...
                #...  de esta manera el tamaño muestreal es igual para las 3 (evitando excesivos outliers)
    
    axes[0].boxplot(
        [data_before[:n_min], data_seizure[:n_min], data_after[:n_min]],
        labels=['Antes', 'Crisis', 'Después']
    )
    axes[0].set_title(f'Diagrama de Cajas - Canal {canal}', fontweight='bold')
    axes[0].set_ylabel('Amplitud de la señal (μV)')
    axes[0].grid(axis='y', linestyle=':', alpha=0.6)

    # --- B. HISTOGRAMA Y PDF en axes[1] ---
    # Graficamos el histograma del bloque 'Crisis' como ejemplo
    counts, bins, patches = axes[1].hist(data_seizure, bins=50, density=True, alpha=0.5, color='tomato', label='Datos empíricos (Crisis)')

    # Ajustamos una PDF Normal (Gaussiana)
    mu, std = norm.fit(data_seizure)
    p_norm = norm.pdf(bins, mu, std)
    axes[1].plot(bins, p_norm, 'k--', linewidth=2, label=f'Normal\n(μ={mu:.2f}, σ={std:.2f})')

    # Ajustamos una PDF t-location-scale (t-Student) sugerida en la teoría
    df, loc, scale = t.fit(data_seizure)
    p_t = t.pdf(bins, df, loc, scale)
    axes[1].plot(bins, p_t, 'b-', linewidth=2, label='t-location-scale')

    axes[1].set_title('Histograma y Ajuste de PDF (Crisis)', fontweight='bold')
    axes[1].legend()

    # --- C. SCATTER PLOT (Diagrama de dispersión de parámetros) en axes[2] ---
    # Extraemos la media y desviación estándar de todos los 28 canales usando tu diccionario 'stats'
    mu_before = stats['before']['mean']
    sigma_before = stats['before']['std']

    mu_seizure = stats['seizure']['mean']
    sigma_seizure = stats['seizure']['std']

    axes[2].scatter(mu_before, sigma_before, color='steelblue', label='0 (Normal)', alpha=0.7)
    axes[2].scatter(mu_seizure, sigma_seizure, color='tomato', label='1 (Crisis)', alpha=0.7)
    axes[2].set_title('Scatter plot classification (μ vs σ)', fontweight='bold')
    axes[2].set_xlabel('Media (μ)')
    axes[2].set_ylabel('Desviación Estándar (σ)')
    axes[2].legend()
    axes[2].grid(linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()
    
    
escenario1paso2()
escenario1paso4()




# Informe Técnico: Análisis Continuo y Detección de Retardo en Señales EEG (Escenario 2)
**Práctica 2 - Algorítmica y Lógica Computacional**

## 1. Caracterización Continua de la Señal (Ventana Deslizante)
Para el Escenario 2, el abordaje metodológico transicionó hacia la evaluación de una señal continua mediante un "Bloque Total" que concatena secuencialmente los periodos de reposo previo (Before), la crisis (Crisis) y la recuperación (After).

Dado que las señales biomédicas son no-estacionarias, se dividió el flujo en **ventanas temporales cortas** (1 a 2 segundos) bajo el supuesto de cuasi-estacionariedad. Se implementó una ventana deslizante que avanza en pasos exactos de la frecuencia de muestreo (1 segundo = fs muestras). Para cada ventana, se extrajeron dinámicamente descriptores como la varianza, desviación estándar, promedio absoluto y autocorrelación, permitiendo trazar la evolución temporal de la actividad neuronal en cada electrodo.

---

## 2. Aplicación de Umbral Dinámico y Tiempo de Retardo
El objetivo es emular un sistema de monitoreo en tiempo real capaz de calcular el **tiempo de retardo de detección**:

*   **Aplicación del Umbral [min, max]:** El algoritmo calculó en las ventanas iniciales (normalidad) los límites de dispersión. Dado que la desviación estándar (σ) ofrece una métrica estandarizada del comportamiento sano, el límite superior (max) se estableció como la cota del umbral.
*   **Determinación del Retardo:** Al avanzar hacia el segmento de crisis, la dispersión registró una escalada abrupta por la actividad neuronal excesiva. El algoritmo identificó la ventana exacta donde el valor fracturó el umbral. Al restar este instante de la marca médica real (anotación clínica), se obtuvo la latencia o inercia matemática del sistema algorítmico.

---

## 3. Dinámica Temporal de la Sincronía y Ritmicidad
*   **Matriz de Correlación de Pearson:** Al trazar la similitud entre canales en el tiempo (rango [-1, 1]), se observó que en reposo los canales operaban de forma ortogonal. En la crisis, se registró un acoplamiento masivo, verificando la **sincronía aumentada** entre distintas regiones cerebrales.
*   **Autocorrelación Dinámica:** Analizando la energía en desplazamientos nulos (lag=0) y unitarios (lag=1), se comprobó que mientras el reposo carece de memoria (similar al ruido blanco), la crisis sostiene valores elevados. Esto confirma la presencia de ritmos patológicos como los patrones de **espiga-onda** (spike-and-wave).

---

## 4. Modelamiento Estadístico Unificado del Bloque Total
*   **Diagrama de Cajas (Boxplot):** La evaluación del bloque total evidenció que el componente patológico estira el "bigote" superior radicalmente por encima de las cercas (fences) estándar, aislando gráficamente las amplitudes extremas de la crisis.
*   **Histograma y PDF:** Se rechazó la distribución Normal debido a los picos y valores aberrantes. El modelo global halló su mejor ajuste en la función **t-location-scale**, donde el parámetro de forma (ν) asimiló con precisión las "colas pesadas" generadas por la actividad epiléptica.
*   **Diagrama de Dispersión (Scatter Plot):** Al extraer el vector de parámetros θ = (μ, σ) por cada ventana, se observó que mientras la media (μ) se mantiene cerca del origen, la desviación estándar (σ) separa drásticamente las ventanas en clústeres, actuando como el principal discriminante (feature) para la clasificación.

---

## 5. Complejidad Algorítmica y el Factor Estocástico
**Análisis de Complejidad (Notación Big-O):**
La implementación mediante ventanas añade un multiplicador lineal. Si la ventana mide W muestras y hay K ventanas en el registro:
1.  **Estadísticos univariados:** Complejidad de **O(K · W)**.
2.  **Relación cruzada (M electrodos):** El costo escala a **O(K · M² · W)**.
3.  **Autocorrelación (vía FFT):** Al optimizar con la Transformada Rápida de Fourier, el costo es de **θ(K · W log₂ W)**, evitando el orden cuadrático que congelaría la ejecución.

**Optimización mediante Muestreo Aleatorio:**
Para evitar procesar la matriz poblacional completa (N), se extrae una Muestra Aleatoria Simple (SRS) de tamaño n. Dado que n << N, la complejidad converge a una carga constante de tipo **O(1)**. Esta aproximación estocástica garantiza un monitoreo ágil y el cálculo instantáneo del retardo en sistemas clínicos de tiempo real sin introducir sesgos sistemáticos.


In [ ]:

"""
Algorítmica y Lógica Computacional — Práctica 2
Escenario 2: Bloque Total con Ventana Deslizante

La señal completa [Before | Crisis | After] se analiza moviéndose en pasos de
frecuencia de muestreo (1 segundo). Cada ventana produce un valor de descriptor
→ se obtiene una serie temporal de descriptores que permite detectar visualmente
cuándo comienza y termina la crisis, y estimar el retardo de detección.

# pip install pyedflib numpy scipy matplotlib
"""

import numpy as np
from scipy import signal
from pyedflib import highlevel
import matplotlib.pyplot as plt
from scipy.stats import norm, t

# =============================================================================
# PARÁMETROS (mismos que arch1.py)
# =============================================================================
EDF_FILE   = './archivos/chb20_12.edf'
START_SEC  = 94          # Segundo donde inicia la crisis
END_SEC    = 123         # Segundo donde termina la crisis
WINDOW_SEC = 120         # Contexto antes y después (2 minutos)

# Tamaño de la ventana deslizante en segundos.
# Con WIN_SEC=1 nos movemos exactamente en pasos de 1 segundo (= fs muestras).
# Se puede agrandar (e.g. 2 o 5 seg) para suavizar los descriptores.
WIN_SEC = 1

# =============================================================================
# CARGA DEL ARCHIVO EDF
# =============================================================================
signals, signal_headers, header = highlevel.read_edf(EDF_FILE)
fs = 256   # Frecuencia de muestreo (Hz) — hardcodeada igual que arch1.py

# Convertir tiempos a muestras
start_sample  = START_SEC  * fs
end_sample    = END_SEC    * fs
window_samples = WINDOW_SEC * fs
total_samples  = signals.shape[1]

inicio_before = max(0, start_sample  - window_samples)
fin_after     = min(total_samples, end_sample + window_samples)

# =============================================================================
# SEGMENTACIÓN Y CENTRADO (idéntico al Escenario 1)
# =============================================================================
def extract_and_center(sig, s, e):
    """Extrae [s:e] de todos los canales y resta la media por canal (elimina DC offset)."""
    segment = sig[:, s:e]
    return segment - segment.mean(axis=1, keepdims=True)

before_centered  = extract_and_center(signals, inicio_before, start_sample)
seizure_centered = extract_and_center(signals, start_sample,  end_sample)
after_centered   = extract_and_center(signals, end_sample,    fin_after)

# ---- Bloque total: [Before | Crisis | After] concatenados ----
total_block = np.concatenate(
    (before_centered, seizure_centered, after_centered), axis=1
)
n_canales, total_len = total_block.shape

print(f"Frecuencia de muestreo : {fs} Hz")
print(f"Bloque total           : {total_block.shape}  "
      f"({total_len / fs:.1f} seg totales)")
print(f"Inicio crisis (muestra): {start_sample - inicio_before}  "
      f"({(start_sample - inicio_before) / fs:.1f} seg dentro del bloque)")
print(f"Fin   crisis (muestra) : {end_sample   - inicio_before}  "
      f"({(end_sample   - inicio_before) / fs:.1f} seg dentro del bloque)")

# =============================================================================
# POSICIONES DE REFERENCIA DENTRO DEL BLOQUE TOTAL
# (para marcar la crisis en los gráficos)
# =============================================================================
# Cuántas muestras hay de "before" antes de la crisis
offset_before = start_sample - inicio_before

seizure_start_in_block = offset_before                    # muestra donde empieza la crisis
seizure_end_in_block   = offset_before + (end_sample - start_sample)  # muestra donde termina

# En segundos (para ejes temporales)
t_seizure_start = seizure_start_in_block / fs
t_seizure_end   = seizure_end_in_block   / fs

# =============================================================================
# VENTANA DESLIZANTE — calcula descriptores moviéndose de 1 segundo en 1 segundo
# =============================================================================
win_len  = WIN_SEC * fs   # longitud de la ventana en muestras
step     = fs             # paso = 1 segundo = fs muestras  (ver consigna: "pasos de fs")

# Calculamos cuántas ventanas enteras caben en el bloque total
n_ventanas = (total_len - win_len) // step + 1

# Arrays donde se guarda el descriptor de CADA CANAL en CADA VENTANA
# shape: (n_canales, n_ventanas)
var_timeline      = np.zeros((n_canales, n_ventanas))
std_timeline      = np.zeros((n_canales, n_ventanas))
abs_mean_timeline = np.zeros((n_canales, n_ventanas))
mean_timeline     = np.zeros((n_canales, n_ventanas))

# Para autocorrelación guardamos solo el valor en lag=0 (energía de la autocorr)
# y en lag=1 segundo (memoria de la señal)
autocorr_lag0_timeline = np.zeros((n_canales, n_ventanas))
autocorr_lag1_timeline = np.zeros((n_canales, n_ventanas))  # lag = fs muestras

# Correlación de Pearson entre canal 0 y canal 1 en cada ventana
pearson_c0c1_timeline  = np.zeros(n_ventanas)

# Tiempo central de cada ventana (en segundos, para el eje x)
t_windows = np.array([(i * step + win_len // 2) / fs for i in range(n_ventanas)])

print(f"\nVentana deslizante: {WIN_SEC} seg  |  Paso: 1 seg  |  Total ventanas: {n_ventanas}")
print("Calculando descriptores por ventana...")

for i in range(n_ventanas):
    s = i * step
    e = s + win_len
    win = total_block[:, s:e]          # shape: (n_canales, win_len)

    var_timeline[:, i]      = np.var(win,   axis=1)
    std_timeline[:, i]      = np.std(win,   axis=1)
    abs_mean_timeline[:, i] = np.mean(np.abs(win), axis=1)
    mean_timeline[:, i]     = np.mean(win,  axis=1)

    # Autocorrelación del canal 0: lag 0 (energía) y lag 1 seg
    ac = signal.correlate(win[0], win[0], mode='full')
    mid = len(ac) // 2
    autocorr_lag0_timeline[0, i] = ac[mid]            # lag = 0
    # lag de 1 segundo (= fs muestras): si existe dentro del array
    lag1_idx = mid + fs
    autocorr_lag1_timeline[0, i] = ac[lag1_idx] if lag1_idx < len(ac) else 0.0

    # Pearson entre canal 0 y canal 1
    pearson_c0c1_timeline[i] = np.corrcoef(win[0], win[1])[0, 1]

print("Listo.")

# =============================================================================
# ESCENARIO 2 — PASO 1 Y 2:
# Graficar descriptores en el tiempo + umbral dinámico + retardo de detección
# =============================================================================
def escenario2_descriptores():
    """
    Grafica varianza, desviación estándar y media absoluta de TODOS los canales
    a lo largo del tiempo, superpone el umbral [min, max] calculado sobre la
    región 'before', y marca el inicio/fin de la crisis.

    Para cada descriptor muestra también el retardo de detección estimado:
    el primer momento en que algún canal supera el umbral.
    """
    descriptores = {
        'VAR (Varianza)'                : var_timeline,
        'STD (Desviación estándar)'     : std_timeline,
        'ABS_MEAN (Media valor absoluto)': abs_mean_timeline,
    }

    # Región "before" dentro de t_windows → ventanas cuyo centro cae antes de la crisis
    before_mask = t_windows < t_seizure_start

    fig, axes = plt.subplots(len(descriptores), 1, figsize=(16, 12), sharex=True)
    fig.suptitle(
        'Escenario 2 — Descriptores estadísticos a lo largo del tiempo\n'
        '(ventana deslizante de 1 seg, todos los canales)',
        fontsize=13, fontweight='bold'
    )

    for ax, (nombre, data) in zip(axes, descriptores.items()):
        # Umbral dinámico: [min, max] del descriptor en la región "before"
        umbral_min = data[:, before_mask].min()
        umbral_max = data[:, before_mask].max()

        # Graficamos cada canal como una línea semitransparente
        for ch in range(n_canales):
            ax.plot(t_windows, data[ch], color='steelblue', alpha=0.25, linewidth=0.7)

        # Banda del umbral "normal"
        ax.axhspan(umbral_min, umbral_max, color='gold', alpha=0.25,
                   label=f'Rango normal [{umbral_min:.2f}, {umbral_max:.2f}]')
        ax.axhline(umbral_max, color='orange', linestyle='--', linewidth=1.2)
        ax.axhline(umbral_min, color='orange', linestyle='--', linewidth=1.2)

        # Marcas de inicio y fin de crisis
        ax.axvline(t_seizure_start, color='red',   linestyle='-',  linewidth=2, label='Inicio crisis')
        ax.axvline(t_seizure_end,   color='darkred', linestyle='-', linewidth=2, label='Fin crisis')

        # Retardo de detección: primer t donde ALGÚN canal supera el umbral_max
        # (solo buscamos después de que empieza la crisis)
        post_onset_mask = t_windows >= t_seizure_start
        over_threshold  = np.any(data[:, post_onset_mask] > umbral_max, axis=0)
        t_post_onset    = t_windows[post_onset_mask]

        if over_threshold.any():
            t_deteccion = t_post_onset[np.argmax(over_threshold)]
            retardo_seg = t_deteccion - t_seizure_start
            ax.axvline(t_deteccion, color='purple', linestyle=':', linewidth=1.8,
                       label=f'Detección (retardo ≈ {retardo_seg:.1f} s)')

        ax.set_ylabel(nombre, fontsize=9)
        ax.legend(loc='upper left', fontsize=8, ncol=2)
        ax.grid(axis='both', linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Tiempo (segundos)', fontsize=11)
    plt.tight_layout()
    plt.show()


# =============================================================================
# ESCENARIO 2 — AUTOCORRELACIÓN Y PEARSON EN EL TIEMPO
# =============================================================================
def escenario2_correlaciones():
    """
    Grafica la autocorrelación (lag 0 y lag 1 seg) del canal 0
    y la correlación de Pearson entre canal 0 y canal 1 en el tiempo.
    Útil para ver la sincronización durante la crisis.
    """
    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
    fig.suptitle(
        'Escenario 2 — Autocorrelación y Correlación de Pearson en el tiempo\n'
        '(canal 0 y par canal 0–canal 1)',
        fontsize=13, fontweight='bold'
    )

    # Umbral before para autocorr lag0
    before_mask = t_windows < t_seizure_start
    umbral_ac0_max = autocorr_lag0_timeline[0, before_mask].max()
    umbral_ac0_min = autocorr_lag0_timeline[0, before_mask].min()

    # --- Autocorr lag 0 (energía) ---
    axes[0].plot(t_windows, autocorr_lag0_timeline[0], color='steelblue', linewidth=1)
    axes[0].axhspan(umbral_ac0_min, umbral_ac0_max, color='gold', alpha=0.25,
                    label=f'Rango normal [{umbral_ac0_min:.0f}, {umbral_ac0_max:.0f}]')
    axes[0].axvline(t_seizure_start, color='red',    linestyle='-', linewidth=2, label='Inicio crisis')
    axes[0].axvline(t_seizure_end,   color='darkred', linestyle='-', linewidth=2, label='Fin crisis')
    axes[0].set_ylabel('Autocorr lag=0 (Canal 0)', fontsize=9)
    axes[0].legend(fontsize=8)
    axes[0].grid(linestyle=':', alpha=0.4)

    # --- Autocorr lag 1 seg ---
    umbral_ac1_max = autocorr_lag1_timeline[0, before_mask].max()
    umbral_ac1_min = autocorr_lag1_timeline[0, before_mask].min()
    axes[1].plot(t_windows, autocorr_lag1_timeline[0], color='seagreen', linewidth=1)
    axes[1].axhspan(umbral_ac1_min, umbral_ac1_max, color='gold', alpha=0.25,
                    label=f'Rango normal [{umbral_ac1_min:.0f}, {umbral_ac1_max:.0f}]')
    axes[1].axvline(t_seizure_start, color='red',    linestyle='-', linewidth=2, label='Inicio crisis')
    axes[1].axvline(t_seizure_end,   color='darkred', linestyle='-', linewidth=2, label='Fin crisis')
    axes[1].set_ylabel('Autocorr lag=1s (Canal 0)', fontsize=9)
    axes[1].legend(fontsize=8)
    axes[1].grid(linestyle=':', alpha=0.4)

    # --- Pearson canal 0 vs canal 1 ---
    umbral_p_max = pearson_c0c1_timeline[before_mask].max()
    umbral_p_min = pearson_c0c1_timeline[before_mask].min()
    axes[2].plot(t_windows, pearson_c0c1_timeline, color='tomato', linewidth=1)
    axes[2].axhspan(umbral_p_min, umbral_p_max, color='gold', alpha=0.25,
                    label=f'Rango normal [{umbral_p_min:.2f}, {umbral_p_max:.2f}]')
    axes[2].axvline(t_seizure_start, color='red',    linestyle='-', linewidth=2, label='Inicio crisis')
    axes[2].axvline(t_seizure_end,   color='darkred', linestyle='-', linewidth=2, label='Fin crisis')
    axes[2].set_ylabel('Pearson C0–C1', fontsize=9)
    axes[2].legend(fontsize=8)
    axes[2].grid(linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Tiempo (segundos)', fontsize=11)
    plt.tight_layout()
    plt.show()


# =============================================================================
# ESCENARIO 2 — PASO 4:
# Histograma + PDF + Diagrama de Cajas + Scatter Plot
# (mismo análisis que Escenario 1 pero sobre el bloque total completo)
# =============================================================================
def escenario2_histograma_cajas():
    """
    Para el bloque total:
      - Diagrama de cajas comparando antes / crisis / después (igual que Escenario 1
        pero acá lo mostramos sobre el total_block para apreciar la diferencia en
        una sola visualización unificada).
      - Histograma + ajuste de PDF Normal y t-location-scale sobre el bloque total
        completo del canal 0.
      - Scatter plot de parámetros (μ, σ) para cada ventana temporal, coloreado
        según si la ventana cae en zona 'before', 'seizure' o 'after'.
    """
    canal = 0

    # Señal completa del canal 0 (usado para histograma)
    data_total = total_block[canal, :]

    # Clasificamos cada ventana temporal en before / seizure / after
    labels_ventana = []
    for t_w in t_windows:
        if t_w < t_seizure_start:
            labels_ventana.append('before')
        elif t_w <= t_seizure_end:
            labels_ventana.append('seizure')
        else:
            labels_ventana.append('after')
    labels_ventana = np.array(labels_ventana)

    mask_b = labels_ventana == 'before'
    mask_s = labels_ventana == 'seizure'
    mask_a = labels_ventana == 'after'

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Escenario 2 — Histograma, PDF y Scatter (bloque total)',
                 fontsize=13, fontweight='bold')

    # --- A. DIAGRAMA DE CAJAS ---
    # Usamos las muestras crudas de cada segmento (como en Escenario 1)
    n_min = min(
        before_centered.shape[1],
        seizure_centered.shape[1],
        after_centered.shape[1]
    )
    axes[0].boxplot(
        [before_centered[canal, :n_min],
         seizure_centered[canal, :n_min],
         after_centered[canal,  :n_min]],
        labels=['Antes', 'Crisis', 'Después'],
        patch_artist=True,
        boxprops=dict(facecolor='lightblue', color='steelblue'),
        medianprops=dict(color='red', linewidth=2)
    )
    axes[0].set_title(f'Diagrama de Cajas — Canal {canal}', fontweight='bold')
    axes[0].set_ylabel('Amplitud (μV)')
    axes[0].grid(axis='y', linestyle=':', alpha=0.6)

    # --- B. HISTOGRAMA + PDF sobre el bloque total del canal 0 ---
    counts, bins, _ = axes[1].hist(
        data_total, bins=60, density=True,
        alpha=0.45, color='slategray', label='Datos totales (Canal 0)'
    )
    # Ajuste Normal
    mu_n, std_n = norm.fit(data_total)
    p_norm = norm.pdf(bins, mu_n, std_n)
    axes[1].plot(bins, p_norm, 'k--', linewidth=2,
                 label=f'Normal (μ={mu_n:.2f}, σ={std_n:.2f})')

    # Ajuste t-location-scale (recomendado en la teoría para señales EEG)
    df_t, loc_t, scale_t = t.fit(data_total)
    p_t = t.pdf(bins, df_t, loc_t, scale_t)
    axes[1].plot(bins, p_t, 'b-', linewidth=2,
                 label=f't-loc-scale (df={df_t:.1f})')

    axes[1].set_title('Histograma + PDF — Bloque Total', fontweight='bold')
    axes[1].set_xlabel('Amplitud (μV)')
    axes[1].legend(fontsize=8)
    axes[1].grid(linestyle=':', alpha=0.4)

    # --- C. SCATTER PLOT de (μ_ventana, σ_ventana) coloreado por etapa ---
    # Cada punto es UNA VENTANA TEMPORAL del canal 0
    mu_v  = mean_timeline[canal, :]
    std_v = std_timeline[canal, :]

    axes[2].scatter(mu_v[mask_b], std_v[mask_b], color='steelblue',
                    label='Before (normal)', alpha=0.6, s=18)
    axes[2].scatter(mu_v[mask_s], std_v[mask_s], color='tomato',
                    label='Crisis', alpha=0.8, s=30)
    axes[2].scatter(mu_v[mask_a], std_v[mask_a], color='seagreen',
                    label='After', alpha=0.6, s=18)

    axes[2].set_title('Scatter μ vs σ por ventana — Canal 0', fontweight='bold')
    axes[2].set_xlabel('Media (μ)')
    axes[2].set_ylabel('Desviación estándar (σ)')
    axes[2].legend(fontsize=9)
    axes[2].grid(linestyle=':', alpha=0.4)

    plt.tight_layout()
    plt.show()



# =============================================================================
# EJECUCIÓN
# =============================================================================
escenario2_descriptores()      # Paso 1 y 2: descriptores en tiempo + umbral + retardo
escenario2_correlaciones()     # Paso 1: autocorrelación y Pearson en tiempo
escenario2_histograma_cajas()  # Paso 4: histograma, PDF, cajas, scatter

